# Hands-on — Aula 02: Pipeline RAG completo

Pipeline de ponta a ponta sobre um corpus fictício de 5 documentos da
"Innovatech" (`data/`), **100% local**:

| Seção | Tema | Slide |
|---|---|---|
| 1 | Ingestão e chunking | 12, 14–15 |
| 2 | Embeddings e indexação vetorial (FAISS) | 16–17, 21–22 |
| 3 | BM25, limites de cada busca e fusão híbrida (RRF) | 20, 23–24 |
| 4 | Geração com grounding, citações e recusa | 25 |
| 5 | Avaliação: recall@k, latência e custo | 26–28 |

**Modelos:** `paraphrase-multilingual-MiniLM-L12-v2` (embeddings) · `Qwen2.5-1.5B-Instruct` (geração)

> Execute as células **na ordem** — cada seção reutiliza o que a anterior construiu.
> Todo o código do pipeline está neste notebook; a pasta `data/` traz os 5
> documentos que fazem o papel da base de conhecimento da empresa.

In [1]:
import json
import re
import time
from pathlib import Path

import numpy as np
import torch
from rank_bm25 import BM25Okapi
from sentence_transformers import SentenceTransformer
from transformers import AutoModelForCausalLM, AutoTokenizer
from transformers.utils import logging as hf_logging

hf_logging.set_verbosity_error()  # silencia avisos verbosos da biblioteca

PASTA_DADOS = Path("data")        # a "base de conhecimento" da empresa fictícia
PASTA_INDICE = Path("indice")     # artefatos que o pipeline vai gerar
MODELO_EMBEDDINGS = "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2"

print("Setup pronto — GPU disponível:", torch.cuda.is_available())

Setup pronto — GPU disponível: True


### As funções do pipeline

Três peças que usaremos ao longo de todo o notebook — leia com atenção,
porque cada uma carrega uma decisão de projeto:

- `dividir_em_chunks`: corte de tamanho fixo com **overlap**, recuando até uma
  fronteira natural (parágrafo ou fim de frase) para não partir uma ideia ao meio;
- `tokenizar_lexical`: a tokenização simples que alimenta o BM25;
- `rrf`: o **Reciprocal Rank Fusion** — a fusão de rankings que será nosso
  re-ranking na busca híbrida, em meia dúzia de linhas.

In [2]:
def dividir_em_chunks(texto, tamanho=400, overlap=80):
    """Chunking de tamanho fixo com overlap e fronteira natural."""
    chunks, inicio = [], 0
    while inicio < len(texto):
        fim = min(inicio + tamanho, len(texto))
        if fim < len(texto):
            janela = texto[inicio:fim]
            corte = max(janela.rfind("\n\n"), janela.rfind(". "))
            if corte > tamanho // 2:           # só recua se sobrar chunk útil
                fim = inicio + corte + 1
        pedaco = texto[inicio:fim].strip()
        if pedaco:
            chunks.append(pedaco)
        if fim == len(texto):
            break
        inicio = fim - overlap                  # o próximo chunk repete o final
    return chunks


def tokenizar_lexical(texto):
    """Tokenização simples para o BM25: minúsculas + palavras alfanuméricas."""
    return re.findall(r"[a-z0-9À-ÿ\-]+", texto.lower())


def rrf(rankings, k=60):
    """Reciprocal Rank Fusion: funde rankings somando 1/(k + posição)."""
    pontos = {}
    for ranking in rankings:
        for posicao, idx in enumerate(ranking):
            pontos[idx] = pontos.get(idx, 0.0) + 1.0 / (k + posicao + 1)
    return sorted(pontos.items(), key=lambda par: -par[1])


print("Funções do pipeline definidas.")

Funções do pipeline definidas.


## Seção 1 — Ingestão e chunking *(slides 12, 14–15)*

Lemos a base de conhecimento e dividimos em chunks de 400 caracteres com
overlap de 80, recuando até uma fronteira natural (parágrafo/frase).
**Observe:** o chunk de exemplo começa no meio de uma palavra — é o overlap
trazendo o final do chunk anterior (gancho para chunking semântico).

In [3]:
print("INGESTÃO — lendo a base de conhecimento (data/*.md)")
documentos = [(arq.stem, arq.read_text(encoding="utf-8"))
              for arq in sorted(PASTA_DADOS.glob("*.md"))]
for nome, texto in documentos:
    print(f"  {nome:22s} {len(texto):5d} caracteres")

print("\nCHUNKING — tamanho 400 caracteres, overlap 80, fronteira natural")
chunks = []
for nome, texto in documentos:
    pedacos = dividir_em_chunks(texto, tamanho=400, overlap=80)
    for i, pedaco in enumerate(pedacos):
        chunks.append({"doc": nome, "chunk_id": f"{nome}#{i}", "texto": pedaco})
    print(f"  {nome:22s} -> {len(pedacos):2d} chunks")
print(f"\nTOTAL: {len(chunks)} chunks")

exemplo = next(c for c in chunks if c["doc"] == "manual-reembolso" and "30 dias" in c["texto"])
print("\nEXEMPLO DE CHUNK (metadados viajam junto até a citação):")
print(f"  doc: {exemplo['doc']}   chunk_id: {exemplo['chunk_id']}")
print(f"  texto: \"{exemplo['texto'][:220]}...\"")

PASTA_INDICE.mkdir(exist_ok=True)
(PASTA_INDICE / "chunks.json").write_text(
    json.dumps(chunks, ensure_ascii=False, indent=1), encoding="utf-8")
print("\nChunks salvos em indice/chunks.json (persistência do índice, slide 17).")

INGESTÃO — lendo a base de conhecimento (data/*.md)
  faq-rh                  1795 caracteres
  manual-reembolso        1554 caracteres
  norma-si-042            1769 caracteres
  politica-viagens        1827 caracteres
  release-produto         1745 caracteres

CHUNKING — tamanho 400 caracteres, overlap 80, fronteira natural
  faq-rh                 ->  7 chunks
  manual-reembolso       ->  6 chunks
  norma-si-042           ->  6 chunks
  politica-viagens       ->  8 chunks
  release-produto        ->  7 chunks

TOTAL: 34 chunks

EXEMPLO DE CHUNK (metadados viajam junto até a citação):
  doc: manual-reembolso   chunk_id: manual-reembolso#1
  texto: "ito e bebidas
alcoólicas nunca são reembolsáveis.

## 2. Prazo para solicitação

O pedido de reembolso deve ser registrado no portal interno em até **30 dias
corridos** após a data da despesa. Pedidos fora desse prazo sã..."

Chunks salvos em indice/chunks.json (persistência do índice, slide 17).


## Seção 2 — Embeddings e indexação vetorial *(slides 16–17, 21–22)*

Cada chunk vira um vetor; o índice permite busca por SIGNIFICADO.
Num corpus deste tamanho a busca é exata; HNSW/ANN importam na escala
(slide 22). Se o FAISS não estiver disponível, caímos para busca exata em
numpy — mesmo resultado.

In [4]:
emb = SentenceTransformer(MODELO_EMBEDDINGS)
vetores = emb.encode([c["texto"] for c in chunks],
                     normalize_embeddings=True, show_progress_bar=False)
print(f"matriz de embeddings: {vetores.shape}")
print(f"  -> {vetores.shape[0]} chunks, cada um virou um vetor de {vetores.shape[1]} dimensões")
np.save(PASTA_INDICE / "embeddings.npy", vetores)

try:
    import faiss
    indice = faiss.IndexFlatIP(vetores.shape[1])  # produto interno = cosseno (normalizados)
    indice.add(vetores)
    print(f"\nINDEXAÇÃO — FAISS IndexFlatIP com {indice.ntotal} vetores")

    def buscar_vetorial(consulta, k=3):
        vetor = emb.encode([consulta], normalize_embeddings=True)
        distancias, posicoes = indice.search(vetor, k)
        return list(zip(posicoes[0], distancias[0]))
except ImportError:
    print("\nINDEXAÇÃO — FAISS indisponível; busca exata em numpy (mesmo resultado)")

    def buscar_vetorial(consulta, k=3):
        vetor = emb.encode([consulta], normalize_embeddings=True)[0]
        sims = vetores @ vetor
        top = np.argsort(-sims)[:k]
        return [(int(i), float(sims[i])) for i in top]

matriz de embeddings: (34, 384)
  -> 34 chunks, cada um virou um vetor de 384 dimensões

INDEXAÇÃO — FAISS IndexFlatIP com 34 vetores


In [5]:
CONSULTA = "Quem precisa autorizar a compra das minhas passagens aéreas?"
print(f'CONSULTA (parafraseada, sem citar "aprovação" nem "gestor"): "{CONSULTA}"')
print("\nTOP-3 POR SIMILARIDADE DE COSSENO:")
for posicao, pontuacao in buscar_vetorial(CONSULTA):
    c = chunks[posicao]
    print(f"  {pontuacao:.3f}  [{c['chunk_id']}]  \"{c['texto'][:90]}...\"")

print("\nA busca vetorial achou o trecho certo POR SIGNIFICADO — as palavras")
print("da pergunta quase não aparecem no documento (slide 21).")

CONSULTA (parafraseada, sem citar "aprovação" nem "gestor"): "Quem precisa autorizar a compra das minhas passagens aéreas?"

TOP-3 POR SIMILARIDADE DE COSSENO:
  0.625  [politica-viagens#3]  "viagens nacionais e
20 dias úteis para internacionais.

## 3. Passagens aéreas

As passage..."
  0.594  [politica-viagens#1]  "ios e prestadores de serviço em atividade pela empresa.

## 2. Aprovação prévia

Toda viag..."
  0.518  [politica-viagens#2]  "pedagem. Viagens internacionais exigem aprovação
adicional da diretoria da área. Solicitaç..."

A busca vetorial achou o trecho certo POR SIGNIFICADO — as palavras
da pergunta quase não aparecem no documento (slide 21).


## Seção 3 — BM25, limites de cada busca e fusão híbrida *(slides 20, 23–24)*

1. Consulta com **sigla exata** → BM25 brilha.
2. Consulta **parafraseada** (o documento diz "trabalho remoto", não "casa")
   → BM25 erra feio; a vetorial acha por significado.
3. **RRF** (Reciprocal Rank Fusion) em ~10 linhas funde os dois rankings e
   acerta os dois casos — é o nosso re-ranking.

In [6]:
bm25 = BM25Okapi([tokenizar_lexical(c["texto"]) for c in chunks])
print("Índice BM25 montado (estatísticas de termos, sem rede neural).")

def ranking_bm25(consulta):
    return list(np.argsort(-bm25.get_scores(tokenizar_lexical(consulta))))

def ranking_vetorial(consulta):
    vetor = emb.encode([consulta], normalize_embeddings=True)[0]
    return list(np.argsort(-(vetores @ vetor)))

def mostrar(titulo, ranking, k=3):
    print(f"\n  {titulo}")
    for posicao in ranking[:k]:
        c = chunks[posicao]
        print(f"    [{c['chunk_id']:22s}]  \"{c['texto'][:70]}...\"")

CONSULTAS = [
    ("Consulta com sigla exata (vocabulário técnico)", "O que a SI-042 exige?"),
    ("Consulta parafraseada (o documento diz 'trabalho remoto', não 'casa')",
     "Posso trabalhar de casa em vez de ir ao escritório?"),
]

for descricao, consulta in CONSULTAS:
    print("=" * 70)
    print(f'{descricao}\nCONSULTA: "{consulta}"')
    r_lex, r_vet = ranking_bm25(consulta), ranking_vetorial(consulta)
    mostrar("BM25 (lexical):", r_lex)
    mostrar("Vetorial (semântica):", r_vet)
    fundido = [posicao for posicao, _ in rrf([r_lex[:10], r_vet[:10]])]
    mostrar("HÍBRIDA (RRF — funde os dois rankings):", fundido)

print("\nO RRF soma 1/(60+posição) de cada lista: quem aparece bem nas duas")
print("sobe ao topo — lexical preserva termos exatos, vetorial cobre paráfrase.")

Índice BM25 montado (estatísticas de termos, sem rede neural).
Consulta com sigla exata (vocabulário técnico)
CONSULTA: "O que a SI-042 exige?"

  BM25 (lexical):
    [norma-si-042#0        ]  "# Norma SI-042 — Segurança da Informação: Credenciais e Acesso

## 1. ..."
    [release-produto#4     ]  "- Tela de administração exibe consumo de licenças em tempo real.

## C..."
    [norma-si-042#5        ]  "time de Segurança, com registro em cofre de senhas.

## 6. Violações e..."

  Vetorial (semântica):
    [norma-si-042#0        ]  "# Norma SI-042 — Segurança da Informação: Credenciais e Acesso

## 1. ..."
    [norma-si-042#5        ]  "time de Segurança, com registro em cofre de senhas.

## 6. Violações e..."
    [release-produto#5     ]  "o fuso horário incorreto em agendamentos criados no modo de
  horário ..."

  HÍBRIDA (RRF — funde os dois rankings):
    [norma-si-042#0        ]  "# Norma SI-042 — Segurança da Informação: Credenciais e Acesso

## 1. ..."
    [norma-si-042#5        

## Seção 4 — Geração com grounding, citações e recusa *(slide 25)*

Recuperamos os top-3 chunks (busca híbrida), montamos o prompt com evidências
numeradas e geramos com o Qwen local. Duas perguntas:
**1.** respondível pela base → resposta com `Fontes: [n]` ·
**2.** fora da base → recusa explícita.

In [7]:
MODELO_GERACAO = "Qwen/Qwen2.5-1.5B-Instruct"
tok = AutoTokenizer.from_pretrained(MODELO_GERACAO)
llm = AutoModelForCausalLM.from_pretrained(
    MODELO_GERACAO,
    dtype=torch.float16 if torch.cuda.is_available() else torch.float32,
    device_map="auto" if torch.cuda.is_available() else None)
print("Qwen carregado — e fica carregado para as próximas células.")

SISTEMA = """Você é um assistente corporativo. Responda APENAS com base nas
evidências numeradas fornecidas, de forma curta e direta.
Termine SEMPRE com uma linha no formato: Fontes: [n]
Se as evidências não contiverem a resposta, diga exatamente:
"Não encontrei essa informação na base de conhecimento." Não invente nada.

Exemplo de resposta:
O limite de hospedagem é definido por cidade na tabela do portal interno.
Fontes: [2]"""

def recuperar(consulta, k=3):
    r_lex = ranking_bm25(consulta)[:10]
    r_vet = ranking_vetorial(consulta)[:10]
    return [chunks[posicao] for posicao, _ in rrf([r_lex, r_vet])[:k]]

def responder(pergunta):
    evidencias = recuperar(pergunta)
    print(f'\nPERGUNTA: "{pergunta}"')
    print("EVIDÊNCIAS RECUPERADAS (top-3 da busca híbrida):")
    blocos = []
    for i, c in enumerate(evidencias, 1):
        print(f"  [{i}] ({c['chunk_id']}) \"{c['texto'][:80]}...\"")
        blocos.append(f"[{i}] (fonte: {c['chunk_id']})\n{c['texto']}")
    usuario = "Evidências:\n\n" + "\n\n".join(blocos) + f"\n\nPergunta: {pergunta}"
    msgs = [{"role": "system", "content": SISTEMA}, {"role": "user", "content": usuario}]
    entrada = tok.apply_chat_template(msgs, add_generation_prompt=True,
                                      return_dict=True, return_tensors="pt").to(llm.device)
    t0 = time.perf_counter()
    with torch.no_grad():
        saida = llm.generate(**entrada, max_new_tokens=180, do_sample=False,
                             pad_token_id=tok.eos_token_id)
    n_prompt = entrada["input_ids"].shape[1]
    print("-" * 70)
    print(f"RESPOSTA ({time.perf_counter() - t0:.1f} s):")
    print(tok.decode(saida[0][n_prompt:], skip_special_tokens=True).strip())

Qwen carregado — e fica carregado para as próximas células.


In [8]:
print("CASO 1 — pergunta respondível pela base")
responder("Quando o reembolso aprovado é pago?")

CASO 1 — pergunta respondível pela base

PERGUNTA: "Quando o reembolso aprovado é pago?"
EVIDÊNCIAS RECUPERADAS (top-3 da busca híbrida):
  [1] (manual-reembolso#4) "egoria são reembolsados apenas até o limite da tabela vigente.

## 5. Pagamento
..."
  [2] (manual-reembolso#5) "salário e
aparece no contracheque como "reembolso de despesas".

## 6. Dúvidas

..."
  [3] (manual-reembolso#1) "ito e bebidas
alcoólicas nunca são reembolsáveis.

## 2. Prazo para solicitação
..."


----------------------------------------------------------------------
RESPOSTA (3.1 s):
O reembolso aprovado é pago na folha do mesmo mês ou, se for posterior ao 15, na folha do mês seguinte. O pagamento é feito em conta salário e aparece no contracheque como "reembolso de despesas". Fontes: [2]


In [9]:
print("CASO 2 — pergunta FORA da base (o correto é recusar)")
responder("Qual é a política de licença maternidade da empresa?")

CASO 2 — pergunta FORA da base (o correto é recusar)

PERGUNTA: "Qual é a política de licença maternidade da empresa?"
EVIDÊNCIAS RECUPERADAS (top-3 da busca híbrida):
  [1] (faq-rh#3) "é feito no
aplicativo corporativo, nos mesmos horários do trabalho presencial.

..."
  [2] (politica-viagens#0) "# Política de Viagens Corporativas — Innovatech Soluções

## 1. Objetivo e abran..."
  [3] (faq-rh#5) "ou casamento.
A inclusão ocorre na virada do mês seguinte.

## Desenvolvimento

..."


----------------------------------------------------------------------
RESPOSTA (0.6 s):
Não encontrei essa informação na base de conhecimento.


## Seção 5 — Avaliação: recall@k, latência e custo *(slides 26–28)*

Gabarito manual de 8 perguntas (pergunta → documento correto). Comparamos a
recuperação **vetorial pura** com a **híbrida**. Se a evidência certa não é
recuperada, nem o melhor LLM salva a resposta — recall@k vem antes de tudo.

In [10]:
# Gabarito: cada pergunta tem resposta em exatamente UM documento da base
PERGUNTAS_TESTE = [
    ("Qual o prazo para solicitar reembolso de uma despesa?", "manual-reembolso"),
    ("O que a norma SI-042 exige para as senhas?", "norma-si-042"),
    ("Quantos dias por semana posso trabalhar de casa?", "faq-rh"),
    ("Quem precisa autorizar a compra das minhas passagens aéreas?", "politica-viagens"),
    ("Quais são as novidades da última versão do InnovaAnalytics?", "release-produto"),
    ("Quando posso voar de classe executiva?", "politica-viagens"),
    ("Como funciona o abono pecuniário?", "faq-rh"),
    ("Depois de quantas tentativas erradas a conta trava?", "norma-si-042"),
]


def docs_do_ranking(ranking, k):
    docs = []
    for posicao in ranking:
        doc = chunks[posicao]["doc"]
        if doc not in docs:
            docs.append(doc)
        if len(docs) == k:
            break
    return docs

def avaliar(nome, funcao_ranking):
    acertos1 = acertos3 = 0
    latencias = []
    print("-" * 70)
    print(f"ESTRATÉGIA: {nome}")
    for pergunta, doc_correto in PERGUNTAS_TESTE:
        t0 = time.perf_counter()
        ranking = funcao_ranking(pergunta)
        latencias.append(time.perf_counter() - t0)
        top1, top3 = docs_do_ranking(ranking, 1), docs_do_ranking(ranking, 3)
        acertos1 += doc_correto in top1
        acertos3 += doc_correto in top3
        situacao = "ok@1" if doc_correto in top1 else ("ok@3" if doc_correto in top3 else "ERROU")
        print(f"  [{situacao:5s}] \"{pergunta[:52]}...\" -> top1={top1[0]}")
    print(f"  recall@1 = {acertos1}/{len(PERGUNTAS_TESTE)}   "
          f"recall@3 = {acertos3}/{len(PERGUNTAS_TESTE)}   "
          f"latência média = {1000 * np.mean(latencias):.0f} ms")

def ranking_hibrido(pergunta):
    return [p for p, _ in rrf([ranking_bm25(pergunta)[:10], ranking_vetorial(pergunta)[:10]])]

print(f"GABARITO: {len(PERGUNTAS_TESTE)} perguntas, cada uma com resposta em 1 documento")
avaliar("VETORIAL pura", ranking_vetorial)
avaliar("HÍBRIDA (BM25 + vetorial + RRF)", ranking_hibrido)

print("\nCUSTO desta avaliação: R$ 0,00 — embeddings e busca 100% locais.")

GABARITO: 8 perguntas, cada uma com resposta em 1 documento
----------------------------------------------------------------------
ESTRATÉGIA: VETORIAL pura
  [ok@1 ] "Qual o prazo para solicitar reembolso de uma despesa..." -> top1=manual-reembolso
  [ok@1 ] "O que a norma SI-042 exige para as senhas?..." -> top1=norma-si-042
  [ok@1 ] "Quantos dias por semana posso trabalhar de casa?..." -> top1=faq-rh
  [ok@1 ] "Quem precisa autorizar a compra das minhas passagens..." -> top1=politica-viagens
  [ok@1 ] "Quais são as novidades da última versão do InnovaAna..." -> top1=release-produto
  [ok@1 ] "Quando posso voar de classe executiva?..." -> top1=politica-viagens
  [ERROU] "Como funciona o abono pecuniário?..." -> top1=politica-viagens
  [ok@3 ] "Depois de quantas tentativas erradas a conta trava?..." -> top1=release-produto
  recall@1 = 6/8   recall@3 = 7/8   latência média = 8 ms
----------------------------------------------------------------------
ESTRATÉGIA: HÍBRIDA (BM25 + vetori